In [1]:
# --- Cell 1: imports + shared config ---
%load_ext autoreload
%autoreload 2
import sys, torch, numpy as np
sys.path.append('..')
sys.path.append('../..')

from common.seed import set_seed
from common.io_utils import save_results, load_results
from task4.img_prep.cifar_data import (make_splits, build_known_loaders,
                                       build_unknown_loaders,
                                       NEAR_UNKNOWN, FAR_UNKNOWN)
from task4.backbones.resnet_cifar import CifarResNet18

set_seed(6304)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_ROOT = '../../datasets'
print(device, torch.cuda.get_device_name(0) if device.type == 'cuda' else '')

cuda NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# --- Cell 2: build and freeze the 90/10 split ---
splits = make_splits(DATA_ROOT, seed=6304, val_fraction=0.1, download=True)
save_results(splits, '../../shared/splits/cifar10_seed6304.json')
print(f"train={len(splits['train_idx'])}  val={len(splits['val_idx'])}")
print(splits['classes'])

100%|██████████| 170M/170M [33:47<00:00, 84.1kB/s]   


train=45000  val=5000
['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
# --- Cell 3: VERIFICATION — split integrity and reproducibility ---
tr = np.array(splits['train_idx']); va = np.array(splits['val_idx'])
print('sizes 45000/5000      :', len(tr) == 45000 and len(va) == 5000)
print('no overlap            :', len(np.intersect1d(tr, va)) == 0)
print('covers all 50000      :', len(np.union1d(tr, va)) == 50000)

from torchvision.datasets import CIFAR10
targets = np.array(CIFAR10(DATA_ROOT, train=True).targets)
tr_counts = np.bincount(targets[tr], minlength=10)
va_counts = np.bincount(targets[va], minlength=10)
print('per-class train       :', tr_counts.tolist())
print('per-class val         :', va_counts.tolist())
print('stratified exactly    :', (tr_counts == 4500).all() and (va_counts == 500).all())

again = make_splits(DATA_ROOT, seed=6304, val_fraction=0.1, download=False)
print('seed reproduces split :', again['train_idx'] == splits['train_idx']
                                 and again['val_idx'] == splits['val_idx'])

sizes 45000/5000      : True
no overlap            : True
covers all 50000      : True
per-class train       : [4500, 4500, 4500, 4500, 4500, 4500, 4500, 4500, 4500, 4500]
per-class val         : [500, 500, 500, 500, 500, 500, 500, 500, 500, 500]
stratified exactly    : True
seed reproduces split : True


In [4]:
# --- Cell 4: VERIFICATION — the fixed unknown groups (labels only, no pixels scored) ---
# This cell reads CIFAR-100 METADATA to confirm the groups are constructed as
# specified. No unknown image passes through any model before notebook 05.
unk = build_unknown_loaders(DATA_ROOT, download=True)
for group, names in (('near', NEAR_UNKNOWN), ('far', FAR_UNKNOWN)):
    g = unk[group]
    per_class = {n: int((np.array(g['names']) == n).sum()) for n in names}
    print(f"{group:4s} n={g['n']:4d}  100 each: {all(v == 100 for v in per_class.values())}")
    print('      ', per_class)
print('near/far disjoint     :', set(NEAR_UNKNOWN).isdisjoint(FAR_UNKNOWN))
print('no name clash with C10:', set(NEAR_UNKNOWN + FAR_UNKNOWN).isdisjoint(splits['classes']))

100%|██████████| 169M/169M [26:07<00:00, 108kB/s]    


near n= 800  100 each: True
       {'bus': 100, 'pickup_truck': 100, 'motorcycle': 100, 'tractor': 100, 'wolf': 100, 'fox': 100, 'leopard': 100, 'camel': 100}
far  n= 800  100 each: True
       {'bottle': 100, 'bowl': 100, 'chair': 100, 'clock': 100, 'keyboard': 100, 'mushroom': 100, 'sunflower': 100, 'wardrobe': 100}
near/far disjoint     : True
no name clash with C10: True


In [5]:
# --- Cell 5: VERIFICATION — CIFAR stem, the layer2 split point, parameter count ---
set_seed(6304)
m = CifarResNet18(num_classes=10).to(device)
print('conv1 kernel/stride   :', m.net.conv1.kernel_size, m.net.conv1.stride)
print('maxpool removed       :', isinstance(m.net.maxpool, torch.nn.Identity))

x = torch.randn(4, 3, 32, 32, device=device)
h = m.forward_pre(x); f = m.forward_post(h); logits, f2 = m(x)
print('forward_pre  shape    :', tuple(h.shape), '(expect (4, 128, 16, 16))')
print('forward_post shape    :', tuple(f.shape), '(expect (4, 512))')
print('logits shape          :', tuple(logits.shape), '(expect (4, 10))')
print('split == full forward :', torch.allclose(f, f2, atol=1e-6))

n_params = sum(p.numel() for p in m.parameters())
print('parameters            :', f'{n_params:,}', '(expect 11,173,962)')
del m, x, h, f, logits, f2; torch.cuda.empty_cache()

conv1 kernel/stride   : (3, 3) (1, 1)
maxpool removed       : True
forward_pre  shape    : (4, 128, 16, 16) (expect (4, 128, 16, 16))
forward_post shape    : (4, 512) (expect (4, 512))
logits shape          : (4, 10) (expect (4, 10))
split == full forward : True
parameters            : 11,173,962 (expect 11,173,962)


In [6]:
# --- Cell 6: VERIFICATION — loaders and normalisation ---
loaders = build_known_loaders(DATA_ROOT, splits, batch_size=128, eval_batch=512,
                              num_workers=0, seed=6304, randaugment=False)
print(f"steps/epoch={loaders['steps_per_epoch']}  (expect 351)")
print(f"n_train={loaders['n_train']} n_val={loaders['n_val']} n_test={loaders['n_test']}")

xb, yb = next(iter(loaders['train']))
print('train batch           :', tuple(xb.shape), 'labels', tuple(yb.shape))
print('batch is even (PROSER):', xb.shape[0] % 2 == 0)

xu, yu = next(iter(unk['near']['loader']))
print(f"known  mean/std       : {xb.mean():.3f} {xb.std():.3f}")
print(f"near   mean/std       : {xu.mean():.3f} {xu.std():.3f}   "
      "(same transform, CIFAR-10 constants on both)")

steps/epoch=351  (expect 351)
n_train=45000 n_val=5000 n_test=10000
train batch           : (128, 3, 32, 32) labels (128,)
batch is even (PROSER): True
known  mean/std       : -0.340 1.129
near   mean/std       : -0.058 1.000   (same transform, CIFAR-10 constants on both)


In [7]:
# --- Cell 7: save the prep record ---
results = {'step': 'prep_data', 'seed': 6304,
           'n_train': loaders['n_train'], 'n_val': loaders['n_val'],
           'n_test': loaders['n_test'], 'steps_per_epoch': loaders['steps_per_epoch'],
           'known_classes': splits['classes'],
           'near_unknown': NEAR_UNKNOWN, 'far_unknown': FAR_UNKNOWN,
           'n_near': unk['near']['n'], 'n_far': unk['far']['n'],
           'splits_file': 'shared/splits/cifar10_seed6304.json',
           'normalisation': 'CIFAR-10 statistics applied to knowns and unknowns alike'}
save_results(results, '../results/prep_data.json')
print('saved')

saved
